In [1]:
import json
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / 'backend' / 'data' / 'market_data.json').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open this notebook from inside the project directory.')
CONFIG_PATH = ROOT / 'backend' / 'data' / 'market_data.json'
DB_PATH = ROOT / 'backend' / 'app' / 'services' / 'price_data' / 'price_data.db'
OUTPUT_DIR = CONFIG_PATH.parent / 'mock_timeseries'
SEED = 42
config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
assets = config['all_assets']
metadata = config['asset_metadata']
assert assets and len(assets) == len(set(assets)), 'Universe must be nonempty and unique.'
assert set(assets) <= set(metadata), 'Missing asset metadata.'
if not DB_PATH.is_file():
    raise FileNotFoundError(f'Existing price database required: {DB_PATH}')
with sqlite3.connect(DB_PATH.as_uri() + '?mode=ro', uri=True) as connection:
    stored = pd.read_sql_query('SELECT * FROM price_history', connection)
missing = set(['Date', *assets]) - set(stored.columns)
if missing:
    raise ValueError(f'Price database missing columns: {sorted(missing)}')
stored['Date'] = pd.to_datetime(stored['Date'], errors='raise')
assert stored['Date'].notna().all() and not stored['Date'].duplicated().any(), 'Invalid/duplicate dates.'
prices = stored.set_index('Date').sort_index().loc[:, assets].astype(float)
has_start, has_end = 'start_date' in config, 'end_date' in config
if has_start != has_end:
    raise ValueError('JSON date range requires both start_date and end_date.')
if has_start:
    start, end = pd.Timestamp(config['start_date']), pd.Timestamp(config['end_date'])
    if pd.isna(start) or pd.isna(end) or start > end or prices.empty:
        raise ValueError('Invalid JSON date range or empty stored prices.')
    if start < prices.index.min() or end > prices.index.max():
        raise ValueError('Stored prices do not cover the JSON date range.')
    prices = prices.loc[start:end]
    date_source = 'JSON bounds, observed price dates only'
else:
    date_source = 'Stored price dates (market_data.json has no date range)'
assert len(prices) >= 30, 'At least 30 observations are needed for trends and checks.'
assert np.isfinite(prices.to_numpy()).all() and (prices > 0).all().all(), 'Invalid source prices; no filling/dropping allowed.'
prices.index.name = 'date'
dates = prices.index
original_prices = prices.copy(deep=True)
returns = prices.pct_change(fill_method=None).iloc[1:]
print(f'Universe ({len(assets)}): {assets}')
print(f'Dates: {dates.min().date()} to {dates.max().date()} ({len(dates)} observations)')
print(f'Date source: {date_source}\nPrice source: {DB_PATH}')
display(prices.head())

Universe (13): ['AAPL', 'AMZN', 'BAC', 'BND', 'GLD', 'GOOGL', 'JNJ', 'JPM', 'MSFT', 'PG', 'TSLA', 'VNQ', 'WMT']
Dates: 2021-03-19 to 2026-03-17 (1254 observations)
Date source: Stored price dates (market_data.json has no date range)
Price source: c:\work\hackathon\backend\app\services\price_data\price_data.db


,AAPL,AMZN,BAC,BND,GLD,GOOGL,JNJ,JPM,MSFT,PG,TSLA,VNQ,WMT
date,,,,,,,,,,,,,
2021-03-19,116.869728,153.748001,34.123352,72.266838,163.240005,100.517227,138.975937,136.620438,221.034576,112.837257,218.289993,75.155952,41.199699
2021-03-22,120.181305,155.543503,33.352856,72.472244,163.000000,100.702202,139.375366,132.948196,226.446457,114.750069,223.333328,75.687515,41.396729
2021-03-23,119.353424,156.875000,32.679779,72.634872,161.809998,101.229836,139.245148,131.618469,227.972183,116.883255,220.720001,75.579529,41.887718
2021-03-24,116.967125,154.353500,32.679779,72.746132,162.369995,100.793434,140.599808,132.639999,225.937927,116.847961,210.089996,75.155952,41.628147
2021-03-25,117.454124,152.313004,33.352856,72.694786,161.779999,100.789970,140.651886,134.339600,222.944077,117.658928,213.463333,75.921181,41.909599


## Generate reproducible mock series

Starting EPS scales are illustrative, not historical facts. Technology estimates have faster baseline growth and larger revision shocks. Quarterly shocks occur every 63 stored observations (an approximate trading quarter). EPS levels accumulate small log revisions; common AR(1) sentiment and ticker-specific AR(1) pressure make consecutive revisions related.

The 10Y series starts at 1.6%, slowly approaches regime targets of 1.6%, 3.7%, and 3.0%, and has modest persistent noise. Bounds of 0.1%-8% are scenario guardrails. Separate random streams isolate EPS from rate changes; each function recreates its stream so rerunning cells gives the same result.

In [2]:
EPS_START = {'AAPL': 6.2, 'AMZN': 2.6, 'BAC': 2.8, 'GOOGL': 5.3,
             'JNJ': 9.5, 'JPM': 11.5, 'MSFT': 8.5, 'PG': 5.5,
             'TSLA': 1.6, 'WMT': 1.8}

def generate_eps(dates, assets, metadata):
    rng = np.random.default_rng(np.random.SeedSequence(SEED).spawn(2)[0])
    n = len(dates)
    regime = np.minimum(np.arange(n) * 3 // n, 2)
    sentiment = np.zeros(n)
    for t in range(1, n):
        sentiment[t] = 0.92 * sentiment[t - 1] + rng.normal(0, 0.00025)
    frames = []
    for ticker in assets:
        applicable = metadata[ticker]['asset_class'] == 'Equity'
        frame = pd.DataFrame({'date': dates, 'ticker': ticker, 'eps_applicable': applicable})
        if not applicable:
            frame['eps_estimate_usd'] = np.nan
        else:
            if ticker not in EPS_START:
                raise ValueError(f'Add an illustrative starting EPS for new equity {ticker}.')
            technology = config['sector_map'][ticker] == 'Technology'
            annual_drift = np.array([0.12, -0.10, 0.10] if technology else [0.06, -0.06, 0.07])
            pressure = 0.0
            log_eps = np.empty(n)
            log_eps[0] = np.log(EPS_START[ticker])
            for t in range(1, n):
                pressure = 0.75 * pressure + rng.normal(0, 0.00035)
                shock = rng.normal(0, 0.035 if technology else 0.02) if t % 63 == 0 else 0.0
                revision = annual_drift[regime[t]] / 252 + sentiment[t] + pressure
                revision += rng.normal(0, 0.0018 if technology else 0.0012) + shock
                log_eps[t] = log_eps[t - 1] + revision
            frame['eps_estimate_usd'] = np.exp(log_eps)
        frame['revision_usd'] = frame['eps_estimate_usd'].diff()
        frame['revision_pct'] = frame['eps_estimate_usd'].pct_change(fill_method=None)
        frames.append(frame)
    return pd.concat(frames, ignore_index=True).sort_values(['date', 'ticker']).reset_index(drop=True)

def generate_rates(dates):
    rng = np.random.default_rng(np.random.SeedSequence(SEED).spawn(2)[1])
    n = len(dates)
    regime = np.minimum(np.arange(n) * 3 // n, 2)
    targets = np.array([1.6, 3.7, 3.0])
    yields = np.empty(n)
    yields[0] = 1.6
    noise = 0.0
    for t in range(1, n):
        noise = 0.35 * noise + rng.normal(0, 0.025)
        yields[t] = np.clip(yields[t - 1] + 0.015 * (targets[regime[t]] - yields[t - 1]) + noise, 0.1, 8.0)
    frame = pd.DataFrame({'date': dates, 'yield_10y_pct': yields})
    frame['change_bp'] = frame['yield_10y_pct'].diff() * 100
    return frame

eps = generate_eps(dates, assets, metadata)
rates = generate_rates(dates)
print(f'Generated {len(eps):,} EPS rows (including non-applicable funds), {len(rates):,} rate rows.')

Generated 16,302 EPS rows (including non-applicable funds), 1,254 rate rows.


In [3]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
datasets = {'eps_estimates_revisions': eps, 'rates_10y': rates,
            'market_prices': prices.reset_index(), 'market_returns': returns.reset_index()}
for name, frame in datasets.items():
    path = OUTPUT_DIR / f'{name}.csv'
    frame.to_csv(path, index=False, date_format='%Y-%m-%d', float_format='%.10g')
    print(f'Saved {len(frame):,} rows: {path}')
    display(frame.head(3))
    display(frame.tail(3))

applicable_eps = eps.loc[eps['eps_applicable']].copy()
print('EPS USD/share: per-security summary')
display(applicable_eps.groupby('ticker')['eps_estimate_usd'].agg(['count', 'min', 'max', 'mean', 'std']))
print('EPS revisions: fractional changes')
display(applicable_eps.groupby('ticker')['revision_pct'].describe())
print('10Y yield (percentage points) and changes (basis points)')
display(rates[['yield_10y_pct', 'change_bp']].describe())
print('Reused prices and derived daily fractional returns')
display(prices.describe())
display(returns.describe())

Saved 16,302 rows: c:\work\hackathon\backend\data\mock_timeseries\eps_estimates_revisions.csv


,date,ticker,eps_applicable,eps_estimate_usd,revision_usd,revision_pct
0,2021-03-19,AAPL,True,6.2,NaN,NaN
1,2021-03-19,AMZN,True,2.6,NaN,NaN
2,2021-03-19,BAC,True,2.8,NaN,NaN


,date,ticker,eps_applicable,eps_estimate_usd,revision_usd,revision_pct
16299,2026-03-17,TSLA,True,1.971881,-0.002711,-0.001373
16300,2026-03-17,VNQ,False,NaN,NaN,NaN
16301,2026-03-17,WMT,True,2.019768,-0.000574,-0.000284


Saved 1,254 rows: c:\work\hackathon\backend\data\mock_timeseries\rates_10y.csv


,date,yield_10y_pct,change_bp
0,2021-03-19,1.600000,NaN
1,2021-03-22,1.631362,3.136236
2,2021-03-23,1.657026,2.566363


...

10Y yield (percentage points) and changes (basis points)


,yield_10y_pct,change_bp
count,1254.000000,1253.000000
mean,2.761644,0.108135
std,0.892669,2.737068
min,1.247839,-8.468596
25%,1.703177,-1.755822
50%,3.007526,0.064468
75%,3.530099,1.958779
max,4.174991,9.248925


Reused prices and derived daily fractional returns


,AAPL,AMZN,BAC,BND,GLD,GOOGL,JNJ,JPM,MSFT,PG,TSLA,VNQ,WMT
count,1254.000000,1254.000000,1254.000000,1254.000000,1254.000000,1254.000000,1254.000000,1254.000000,1254.000000,1254.000000,1254.000000,1254.000000,1254.000000
mean,184.943547,165.989021,37.242024,69.162573,222.660255,154.576253,155.980342,180.509028,350.359123,143.609529,272.149389,83.304362,64.414205
std,41.666881,42.117384,7.747679,3.570911,77.074363,56.014560,20.234902,65.788140,87.293108,14.104625,82.780067,7.452145,24.730335
min,116.782066,81.820000,23.666023,61.541416,151.229996,82.746101,135.133759,94.269760,208.183914,112.837257,108.099998,64.952866,37.706764
25%,149.087814,131.932499,31.365420,66.174786,169.792496,118.093199,145.400040,131.679176,272.884941,132.288647,211.875004,77.210663,45.206606
50%,175.438004,168.961998,36.609257,69.185822,183.805000,139.107040,150.358276,147.525581,329.445404,143.248085,251.445000,85.088673,51.400171
75%,219.374798,195.952496,42.399134,72.943190,247.247498,170.659500,157.798847,232.392532,416.078102,155.695614,328.497498,89.384741,86.498850
max,285.922455,254.000000,56.928177,74.971725,495.899994,343.448242,248.559998,334.609985,539.825256,173.840240,489.880005,99.132187,133.889999


,AAPL,AMZN,BAC,BND,GLD,GOOGL,JNJ,JPM,MSFT,PG,TSLA,VNQ,WMT
count,1253.000000,1253.000000,1253.000000,1253.000000,1253.000000,1253.000000,1253.000000,1253.000000,1253.000000,1253.000000,1253.000000,1253.000000,1253.000000
mean,0.000770,0.000514,0.000404,0.000026,0.000885,0.001088,0.000485,0.000710,0.000608,0.000296,0.001172,0.000243,0.000975
std,0.017314,0.022187,0.016928,0.003769,0.010878,0.019330,0.010511,0.015326,0.016472,0.010996,0.037244,0.011830,0.013268
min,-0.092456,-0.140494,-0.110633,-0.016153,-0.102742,-0.095094,-0.075917,-0.074838,-0.099931,-0.062322,-0.154262,-0.049989,-0.113758
25%,-0.007737,-0.011280,-0.009075,-0.002206,-0.004643,-0.009534,-0.005250,-0.007373,-0.007671,-0.005648,-0.019826,-0.006423,-0.005689
50%,0.001140,0.000423,0.000370,0.000000,0.000754,0.001419,0.000401,0.001163,0.000679,0.000651,0.001231,0.000339,0.001081
75%,0.009607,0.012791,0.009269,0.002277,0.006645,0.011301,0.006040,0.009112,0.009904,0.006872,0.020970,0.007095,0.007730
max,0.153288,0.135359,0.084288,0.020702,0.063587,0.102244,0.061932,0.115445,0.101337,0.042699,0.226900,0.073147,0.095489


In [5]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

PLOT_TICKER = 'AAPL'
aapl_eps = eps.loc[eps['ticker'].eq(PLOT_TICKER)].set_index('date')['eps_estimate_usd']
comparison = pd.DataFrame({
    'AAPL price (stored)': prices[PLOT_TICKER],
    'AAPL EPS estimate (mock)': aapl_eps,
    '10Y yield (mock)': rates.set_index('date')['yield_10y_pct'],
}).reindex(dates)
assert np.isfinite(comparison.to_numpy()).all(), 'Plot series must cover the full horizon.'

comparison_chart = make_subplots(
    rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.08,
    subplot_titles=list(comparison.columns),
)
units = ['USD/share', 'USD/share', '%']
for row, (column, unit) in enumerate(zip(comparison.columns, units), start=1):
    comparison_chart.add_trace(go.Scatter(
        x=comparison.index, y=comparison[column], name=column, mode='lines',
        hovertemplate=f'%{{x|%Y-%m-%d}}<br>%{{y:.3f}} {unit}<extra>%{{fullData.name}}</extra>',
    ), row=row, col=1)
    comparison_chart.update_yaxes(title_text=unit, row=row, col=1)
comparison_chart.update_xaxes(range=[dates.min(), dates.max()])
comparison_chart.update_xaxes(title_text='Date', row=3, col=1)
comparison_chart.update_layout(
    title='AAPL price, mock EPS estimate, and mock 10Y yield (original units)',
    template='plotly_white', hovermode='x unified', height=900, showlegend=False,
)
comparison_chart.show()

## Sanity checks

Check exact universe/date alignment, expected missing values, positive levels, revision arithmetic, persistence, unchanged source prices, reproducibility, and saved CSV round trips. These checks establish consistency and realistic-looking persistence, **not empirical calibration**. The initial return row is omitted; initial EPS revisions and rate changes remain empty.

In [ ]:
assert len(eps) == len(dates) * len(assets)
assert set(eps['ticker']) == set(assets)
assert not eps.duplicated(['date', 'ticker']).any()
numeric_eps = ['eps_estimate_usd', 'revision_usd', 'revision_pct']
for ticker, group in eps.groupby('ticker'):
    pd.testing.assert_index_equal(pd.DatetimeIndex(group['date'], name='date'), dates)
    expected_applicable = metadata[ticker]['asset_class'] == 'Equity'
    assert group['eps_applicable'].eq(expected_applicable).all()
    if expected_applicable:
        assert np.isfinite(group['eps_estimate_usd']).all() and group['eps_estimate_usd'].gt(0).all()
        assert group[['revision_usd', 'revision_pct']].iloc[0].isna().all()
        assert np.isfinite(group[['revision_usd', 'revision_pct']].iloc[1:]).all().all()
        np.testing.assert_allclose(group['revision_usd'], group['eps_estimate_usd'].diff(), equal_nan=True)
        np.testing.assert_allclose(group['revision_pct'], group['eps_estimate_usd'].pct_change(fill_method=None), equal_nan=True)
        assert group['eps_estimate_usd'].autocorr() > 0.8, f'Unexpectedly low EPS persistence: {ticker}'
    else:
        assert group[numeric_eps].isna().all().all()
pd.testing.assert_index_equal(pd.DatetimeIndex(rates['date'], name='date'), dates)
assert np.isfinite(rates['yield_10y_pct']).all()
assert rates['yield_10y_pct'].between(0.1, 8.0).all()
assert rates['yield_10y_pct'].nunique() > 1
assert not rates['yield_10y_pct'].isin([0.1, 8.0]).all()
assert pd.isna(rates['change_bp'].iloc[0]) and np.isfinite(rates['change_bp'].iloc[1:]).all()
np.testing.assert_allclose(rates['change_bp'], rates['yield_10y_pct'].diff() * 100, equal_nan=True)
rate_persistence = rates['yield_10y_pct'].autocorr()
assert rate_persistence > 0.8
pd.testing.assert_frame_equal(prices, original_prices)
pd.testing.assert_frame_equal(returns, prices.pct_change(fill_method=None).iloc[1:])
assert np.isfinite(returns.to_numpy()).all()
pd.testing.assert_frame_equal(eps, generate_eps(dates, assets, metadata))
pd.testing.assert_frame_equal(rates, generate_rates(dates))
for name, frame in datasets.items():
    saved = pd.read_csv(OUTPUT_DIR / f'{name}.csv', parse_dates=['date'])
    assert list(saved.columns) == list(frame.columns) and len(saved) == len(frame)
    pd.testing.assert_series_equal(saved['date'], frame['date'].reset_index(drop=True))
    for column in frame.columns.drop('date'):
        expected = frame[column].reset_index(drop=True)
        if pd.api.types.is_numeric_dtype(expected) and not pd.api.types.is_bool_dtype(expected):
            np.testing.assert_allclose(saved[column], expected, rtol=1e-8, atol=1e-10, equal_nan=True)
        else:
            pd.testing.assert_series_equal(saved[column], expected)
print(f'10Y lag-one autocorrelation: {rate_persistence:.4f}')
print('EPS lag-one autocorrelation:')
display(applicable_eps.groupby('ticker')['eps_estimate_usd'].agg(lambda series: series.autocorr()).rename('lag_one_autocorrelation'))
print('All sanity checks passed: alignment, missingness, arithmetic, persistence, reproducibility, and CSV round trips.')